# Interfaces

## Un protocole et deux classes

Définissez le protocole et les classes suivants :

- Le protocole s'appelle `Agregateur` et demande une méthode `reduit(self, elements: list[int]) -> float`
- `Somme`, qui n'hérite de rien, implémente `reduit` par la somme
- `Minimum`, qui n'hérite de rien non plus, implémente `reduit` par le minimum. Elle accepte un paramètre `cle` dans sa méthode `__init__`

Testez ce code avec une liste que vous aurez créée.

In [ ]:
# Votre code ici

### Conseils supplémentaires

Le corps d'une méthode de protocole est `...`. `Somme` et `Minimum` sont des classes ordinaires, sans parent : elles se conforment au protocole par leur seule méthode `reduit`. `Minimum` garde `cle` dans `self.cle`, et le passe à `min(elements, key=self.cle)`.

Avec `donnees = [1, 2, 3]`, `Minimum().reduit(donnees)` renvoie `1`, `Somme().reduit(donnees)` renvoie `6`, et `Minimum(cle=lambda x: -x).reduit(donnees)` renvoie `3`.

```python
from typing import Protocol


class Agregateur(Protocol):
  def reduit(self, elements: list[int]) -> float: ...


class Somme:
  def reduit(self, elements: list[int]) -> float: ...


class Minimum:
  def __init__(self, cle=None): ...

  def reduit(self, elements: list[int]) -> float: ...
```

### Solution

In [ ]:
from typing import Protocol


class Agregateur(Protocol):
  def reduit(self, elements: list[int]) -> float: ...


class Somme:
  def reduit(self, elements: list[int]) -> float:
    return sum(elements)


class Minimum:
  def __init__(self, cle=None):
    self.cle = cle

  def reduit(self, elements: list[int]) -> float:
    return min(elements, key=self.cle)


donnees = [1, 2, 3]
print(Minimum().reduit(donnees))
print(Somme().reduit(donnees))
print(Minimum(cle=lambda x: -x).reduit(donnees))

## Coder pour le protocole

Écrivez une fonction cliente `resumer(agregateur: Agregateur, valeurs: list[int]) -> str` qui renvoie un texte comme `"3 valeurs : 6"` : le nombre de valeurs, puis le résultat de `reduit`. Elle ne doit connaître que le protocole, ni `Somme` ni `Minimum`.

Appelez-la avec `Somme()` et `Minimum()`. Ajoutez ensuite une troisième classe, `Moyenne`, et vérifiez que `resumer` marche sans y toucher.

In [ ]:
# Votre code ici

### Conseils supplémentaires

`resumer` n'appelle que `agregateur.reduit(valeurs)`, et `len(valeurs)` pour le nombre de valeurs. La méthode `reduit` de `Moyenne` divise la somme par le nombre d'éléments.

Avec `donnees = [1, 2, 3]`, `resumer(Moyenne(), donnees)` renvoie `"3 valeurs : 2.0"`.

```python
def resumer(agregateur: Agregateur, valeurs: list[int]) -> str:
  return f"{len(valeurs)} valeurs : {...}"


class Moyenne:
  def reduit(self, elements: list[int]) -> float: ...
```

### Solution

In [ ]:
def resumer(agregateur: Agregateur, valeurs: list[int]) -> str:
  return f"{len(valeurs)} valeurs : {agregateur.reduit(valeurs)}"


class Moyenne:
  def reduit(self, elements: list[int]) -> float:
    return sum(elements) / len(elements)


print(resumer(Somme(), donnees))
print(resumer(Minimum(), donnees))
print(resumer(Moyenne(), donnees))

## Ce que Python ne vérifie pas

Que pouvez-vous dire de la classe suivante ? Que se passe-t-il à l'appel de `resumer(Mediane(), donnees)` ?

In [ ]:
class Mediane:
  def reduction(self, elements: list[int]) -> float:
    return sorted(elements)[len(elements) // 2]

In [ ]:
# Votre code ici

### Conseils supplémentaires

`Mediane` a `reduction`, pas `reduit`. L'appel lève une exception : attrapez-la, pour que le carnet continue.

```python
try:
  resumer(Mediane(), donnees)
except AttributeError as erreur:
  ...
```

### Solution

In [ ]:
# Python ne regarde pas l'annotation Agregateur : l'erreur n'arrive qu'à l'appel de
# reduit, dans resumer, et rien ne l'annonce avant.
try:
  resumer(Mediane(), donnees)
except AttributeError as erreur:
  print("AttributeError :", erreur)

## Le vérificateur de types

Les cellules suivantes installent `pyrefly` et écrivent ce code dans un fichier. Lancez `pyrefly` sur ce fichier, lisez son erreur, puis corrigez `Mediane` jusqu'à ce qu'il ne signale plus rien.

**Documentation utile**

- [Pyrefly, configuration](https://pyrefly.org/en/docs/configuration/)

In [ ]:
!pip install -q pyrefly

In [ ]:
%%writefile agregateurs.py
from typing import Protocol


class Agregateur(Protocol):
  def reduit(self, elements: list[int]) -> float: ...


class Mediane:
  def reduction(self, elements: list[int]) -> float:
    return sorted(elements)[len(elements) // 2]


def resumer(agregateur: Agregateur, valeurs: list[int]) -> str:
  return f"{len(valeurs)} valeurs : {agregateur.reduit(valeurs)}"


print(resumer(Mediane(), [1, 2, 3]))

In [ ]:
# Votre code ici

### Conseils supplémentaires

Sans fichier de configuration, `pyrefly` applique un préréglage minimal qui ne dit rien ici : passez-lui `--preset default`. Il signale la ligne de l'appel à `resumer`, avant toute exécution. Corrigez en renommant `reduction`, avec `%%writefile` ou dans l'éditeur, puis relancez.

```python
!pyrefly check --preset default agregateurs.py
```

### Solution

In [ ]:
!pyrefly check --preset default agregateurs.py

In [ ]:
%%writefile agregateurs.py
from typing import Protocol


class Agregateur(Protocol):
  def reduit(self, elements: list[int]) -> float: ...


class Mediane:
  def reduit(self, elements: list[int]) -> float:
    return sorted(elements)[len(elements) // 2]


def resumer(agregateur: Agregateur, valeurs: list[int]) -> str:
  return f"{len(valeurs)} valeurs : {agregateur.reduit(valeurs)}"


print(resumer(Mediane(), [1, 2, 3]))

In [ ]:
!pyrefly check --preset default agregateurs.py

## Une classe qu'on ne peut pas modifier

Définissez un protocole `Fermable`, qui demande une méthode `close(self) -> None`, et une fonction `ferme_tout(objets: list[Fermable]) -> None` qui les ferme tous.

Appelez-la avec un `io.StringIO` et un fichier ouvert : leurs classes viennent de la bibliothèque standard, et ne connaissent pas votre protocole. Affichez leur attribut `closed` pour vérifier.

In [ ]:
# Votre code ici

### Conseils supplémentaires

`ferme_tout` parcourt la liste et appelle `close` sur chaque objet. Un fichier ouvert : `open("notes.txt", "w", encoding="utf-8")`. Après l'appel, `closed` vaut `True` pour les deux.

```python
from io import StringIO
from typing import Protocol


class Fermable(Protocol):
  def close(self) -> None: ...


def ferme_tout(objets: list[Fermable]) -> None: ...
```

### Solution

In [ ]:
from io import StringIO


class Fermable(Protocol):
  def close(self) -> None: ...


def ferme_tout(objets: list[Fermable]) -> None:
  for objet in objets:
    objet.close()


tampon = StringIO("un texte")
fichier = open("notes.txt", "w", encoding="utf-8")
ferme_tout([tampon, fichier])
print(tampon.closed, fichier.closed)

## `@runtime_checkable` et ses limites

Considérons la classe suivante :

In [ ]:
class Brouillon:
  def reduit(self) -> float:
    return 0.0

Que donne `isinstance(Somme(), Agregateur)` ?

Définissez ensuite `AgregateurVerifiable`, le même protocole décoré de `@runtime_checkable`, et comparez `isinstance` sur `Somme()`, `Mediane()` et `Brouillon()`. Qu'est-ce que `isinstance` ne vérifie pas ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

Sans le décorateur, `isinstance` lève une `TypeError`, à attraper. Avec lui, vous devez obtenir `True`, `False`, puis `True` : `Brouillon` a bien une méthode `reduit`, mais sans paramètre.

```python
from typing import runtime_checkable


@runtime_checkable
class AgregateurVerifiable(Protocol):
  def reduit(self, elements: list[int]) -> float: ...
```

### Solution

In [ ]:
from typing import runtime_checkable

try:
  isinstance(Somme(), Agregateur)
except TypeError as erreur:
  print("TypeError :", erreur)


@runtime_checkable
class AgregateurVerifiable(Protocol):
  def reduit(self, elements: list[int]) -> float: ...


print(isinstance(Somme(), AgregateurVerifiable))
print(isinstance(Mediane(), AgregateurVerifiable))

# Brouillon passe : isinstance regarde si la méthode existe, pas sa signature. Seul le
# vérificateur de types voit que reduit n'y prend pas de paramètre.
print(isinstance(Brouillon(), AgregateurVerifiable))

## Un protocole de la bibliothèque standard (facultatif)

`collections.abc` offre des protocoles tout faits. Écrivez une classe `Panier` qui a une méthode `__len__`, sans parent, et une fonction `est_vide(objet: Sized) -> bool`. Essayez-la avec `Panier` et une liste, puis vérifiez `isinstance(Panier([]), Sized)`.

**Documentation utile**

- [`collections.abc.Sized`](https://docs.python.org/fr/3/library/collections.abc.html#collections.abc.Sized)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`Sized` s'importe de `collections.abc`. Il accepte `isinstance` sans décorateur : `isinstance(Panier([]), Sized)` renvoie `True`, sans héritage.

```python
from collections.abc import Sized


class Panier:
  def __init__(self, articles: list[str]) -> None:
    self.articles = articles

  def __len__(self) -> int: ...


def est_vide(objet: Sized) -> bool: ...
```

### Solution

In [ ]:
from collections.abc import Sized


class Panier:
  def __init__(self, articles: list[str]) -> None:
    self.articles = articles

  def __len__(self) -> int:
    return len(self.articles)


def est_vide(objet: Sized) -> bool:
  return len(objet) == 0


print(est_vide(Panier([])), est_vide([1]))
print(isinstance(Panier([]), Sized))